# SEM → SAM → Parametric Fit → Spatial Compare — Worked Example

This notebook runs the full pipeline in [`sem_pipeline.py`](sem_pipeline.py) on a real
SEM photo (`data/SEM_results.tif`) one stage at a time, so you can see what each
step produces.

**Pipeline stages**

1. **SEM → scale** — OCR the scale-bar label and measure its pixel length → `nm/px`.
2. **SEM → cutout (SAM)** — segment the structure from click points → contour in nm.
3. **Optimize fit** — fit the parametric waveguide+taper shape: search shape params
   (outer) and, for each, the best translation **and rotation/angle** for overlap (inner).
4. **Compare vs existing** — spatially register the fitted outline onto an existing
   design GDS by maximizing overlap area; write a combined GDS.

**Before running:** you need the SAM checkpoint. This folder ships a symlink
`sam_vit_b_01ec64.pth → ../old/sam_vit_b_01ec64.pth`. If that link is broken, download it:

```bash
wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
```

and install deps with `pip install -r requirements.txt` (plus the system `tesseract` binary).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
%matplotlib inline

import sem_pipeline as sp   # the consolidated pipeline module

SEM_PATH  = 'data/SEM_results.tif'   # the SEM photo
SAM_CKPT  = 'sam_vit_b_01ec64.pth'   # SAM weights (symlinked into this folder)
DESIGN_GDS = 'data/my_layout.gds'    # an existing design layout to compare against

img = np.array(Image.open(SEM_PATH).convert('RGB'))
print('image size (W x H):', img.shape[1], 'x', img.shape[0])

## Stage 1 — SEM → scale (nm per pixel)

`detect_scale_nm_per_px` OCRs the scale-bar text (`text_box` ROI) to get the physical
length, measures the bar's pixel span (`scale_box` ROI) by locating its two bright
end-caps, and divides. The default ROIs are tuned for this image's chrome; for a
different microscope/magnification, pass new `text_box`/`scale_box` pixel rectangles
— or skip this stage entirely by supplying a known `nm_per_px` downstream.

*Expected: OCR `'300 nm'` over ~68 px → ~4.41 nm/px.*

In [ ]:
scale = sp.detect_scale_nm_per_px(SEM_PATH)
nm_per_px = scale['nm_per_px']
scale

In [ ]:
# Visualize the two ROIs the detector reads from.
from matplotlib.patches import Rectangle
fig, ax = plt.subplots(figsize=(11, 8)); ax.imshow(img)
for box, color, label in [(sp.DEFAULT_TEXT_BOX, 'yellow', 'text ROI (OCR)'),
                          (sp.DEFAULT_SCALE_BOX, 'cyan', 'scale-bar ROI')]:
    x1, y1, x2, y2 = box
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, ec=color, fc='none', lw=2, label=label))
ax.legend(loc='upper right'); ax.set_title(f"Scale detection → {nm_per_px:.3f} nm/px"); ax.axis('off');

## Stage 2 — SEM → cutout with SAM

We load Segment Anything once, give it a couple of **positive** click points inside the
bright structure, and take the largest contour of the returned mask — converted to
nanometers using the Stage-1 scale.

Here we supply the points programmatically so the notebook runs headless. For manual
clicking instead, call `sp.sam_cutout(SEM_PATH, nm_per_px, points=None)` from a script
(it opens an interactive window).

*Loading the 358 MB checkpoint takes a few seconds (uses Apple MPS if available).*

In [ ]:
predictor = sp.load_sam_predictor(SAM_CKPT)   # slow: loads the model once, reuse it

sam_points = [[400, 190], [900, 190]]         # two points inside the bright structure
contour_nm = sp.sam_cutout(SEM_PATH, nm_per_px, predictor=predictor, points=sam_points)
print('contour points:', len(contour_nm), '| x-extent (nm):',
      round(np.ptp(contour_nm[:,0]), 1))

In [ ]:
# Overlay the extracted contour (convert nm back to px for display).
cpx = contour_nm / nm_per_px
fig, ax = plt.subplots(figsize=(11, 8)); ax.imshow(img)
ax.plot(cpx[:,0], cpx[:,1], 'r-', lw=2, label='SAM contour')
ax.scatter([p[0] for p in sam_points], [p[1] for p in sam_points],
           c='lime', marker='x', s=120, label='click points')
ax.legend(loc='upper right'); ax.set_title('Stage 2 — SAM cutout'); ax.axis('off');

## Stage 3 — Optimize fit (shape params + angle for overlap)

Stage 3 fits an idealized **parametric** shape to the measured contour by searching
(a) the shape parameters and (b) for each candidate, the best rigid placement —
translation `(dx,dy)` **and rotation `θ`** — that maximizes overlap.

**Model choice matters.** The pipeline ships two parametric models:

* `generate_waveguide_shape` — a straight waveguide + a single one-sided taper
  (`[W1,W2,W3,W4,L1,L2,length_waveguide]`). Good for a guide-plus-taper device.
* `generate_diamond_device` — a **symmetric diamond taper + guide with a step**
  (`[w_tip, w_max, w_neck, w_guide, L_up, L_down, L_guide]`). It widens from a narrow
  tip (`w_tip`) to a peak (`w_max`), narrows to a neck (`w_neck`), then a **vertical step
  edge** drops it to the guide width (`w_guide`). This matches the device in this SEM;
  the one-sided template cannot (it would collapse to a thin rod with a huge Hausdorff).

`estimate_diamond_nominal` reads the contour's width profile `W(x)` to auto-seed the
nominal parameters (including locating the step); `diamond_bounds` builds ±35% (widths) /
±30% (lengths) search boxes.

Fitting is **nested**:

* **Inner** (`fit_waveguide_to_contours`, `shape_fn=`): for *fixed* shape params, find the
  best `(dx,dy,θ)` (centroid align → 0/90/180/270° sweep → per-sector ±45° refine).
  Returns the Hausdorff distance (nm).
* **Outer** (`optimize_shape_params`): differential evolution over the shape params,
  calling the inner search for each candidate.

First a **single inner fit** on the auto-estimated nominal — fast (~2 s) and already a
close match.

In [ ]:
# Fit the diamond model that matches this device's topology.
shape_fn = sp.generate_diamond_device
nominal  = sp.estimate_diamond_nominal(contour_nm)   # auto-seed from the width profile
bounds   = sp.diamond_bounds(nominal)                # ±35% widths / ±30% lengths
print('estimated nominal:',
      {n: round(v, 0) for n, v in zip(sp.DIAMOND_PARAM_NAMES, nominal)})

# Single inner placement fit on the nominal params (fast).
hausdorff, aligned = sp.fit_waveguide_to_contours(contour_nm, nominal, shape_fn=shape_fn)
print(f'Hausdorff distance: {hausdorff:.1f} nm')

In [ ]:
# Measured contour vs. the placed parametric shape (in nm).
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(contour_nm[:,0], contour_nm[:,1], 'b-', lw=2, label='SAM contour (measured)')
ax.plot(aligned[:,0], aligned[:,1], 'r--', lw=2, label='fitted parametric shape')
ax.set_aspect('equal'); ax.legend(); ax.set_xlabel('x (nm)'); ax.set_ylabel('y (nm)')
ax.set_title(f'Stage 3 (inner fit) — Hausdorff {hausdorff:.1f} nm'); ax.grid(alpha=0.3);

### Full outer optimization (optional, slow)

This searches all shape params, running the inner placement fit for every candidate —
**a few minutes** (each evaluation is ~2 s and DE does many). Because the nominal is
auto-estimated from the contour, the single fit above is already close; the outer search
mainly tightens the tip width and segment lengths. Set `RUN_FULL_OPT = True` to run it.

In [ ]:
RUN_FULL_OPT = False   # flip to True to run the (slow) global shape search

if RUN_FULL_OPT:
    fit = sp.optimize_shape_params(contour_nm, nominal, shape_fn=shape_fn, bounds=bounds,
                                   popsize=6, maxiter=15)
    best_params = fit['best_shape_params']
    print('best Hausdorff:', round(fit['best_hausdorff'], 1))
    print({n: round(float(v), 1) for n, v in zip(sp.DIAMOND_PARAM_NAMES, best_params)})
else:
    best_params = nominal   # use the auto-estimated nominal for the rest of the notebook
    print('Skipping full optimization; using estimated nominal for downstream demo.')

## Stage 4 — Compare against an existing design (spatial registration)

Given the fitted outline and an existing **design** layout (`data/my_layout.gds`),
`superimpose_gds` registers the outline onto the design by **maximizing intersection
area** (centroid align → angle grid in [0,180]° with a per-angle `(dx,dy)` search → a
final 3-D polish), then writes a combined GDS: design on layer 1, aligned outline on
layer 2.

We first write a single-polygon outline GDS to compare (the fitted shape). Settings are
reduced here for speed; the function defaults (`angle_steps=37`, larger populations) are
more thorough.

In [ ]:
import gdspy

# Recompute the placed outline for the chosen params and save it as a 1-polygon GDS.
_, outline_shape = sp.fit_waveguide_to_contours(contour_nm, best_params, shape_fn=shape_fn)
lib = gdspy.GdsLibrary(); cell = lib.new_cell('OUTLINE')
cell.add(gdspy.Polygon(outline_shape, layer=1)); lib.write_gds('outline_only.gds')

compare = sp.superimpose_gds('outline_only.gds', DESIGN_GDS, 'combined.gds',
                             angle_steps=9, pop_xy=6, iters_xy=15, pop_final=8, iters_final=20)
print({k: (round(v,1) if isinstance(v, float) else v)
       for k, v in compare.items() if k != 'centroid_offset'})

In [ ]:
# Plot the combined result: design polygons (layer 1) + aligned outline (layer 2).
lib = gdspy.GdsLibrary(); lib.read_gds('combined.gds'); top = lib.top_level()[0]
fig, ax = plt.subplots(figsize=(10, 7))
for poly, layer in zip(top.polygons, [p.layers[0] for p in top.polygons]):
    pts = poly.polygons[0]
    color = 'tab:blue' if layer == 1 else 'tab:red'
    ax.fill(pts[:,0], pts[:,1], color=color, alpha=0.35)
    ax.plot(np.r_[pts[:,0], pts[0,0]], np.r_[pts[:,1], pts[0,1]], color=color, lw=1.2)
ax.set_aspect('equal'); ax.set_title('Stage 4 — outline (red) registered onto design (blue)')
ax.set_xlabel('x (nm)'); ax.set_ylabel('y (nm)'); ax.grid(alpha=0.3);

## One call: `run_full_pipeline`

Everything above is wrapped in a single function. It runs scale → SAM → fit → (compare),
and saves `<out_prefix>.gds`, `<out_prefix>.json`, and `<out_prefix>_combined.gds`.
Pass `sam_points=None` for interactive clicking, or `nm_per_px=<float>` to skip OCR.

It runs the **full** outer optimization, so expect it to take a few minutes.

In [ ]:
# result = sp.run_full_pipeline(
#     sem_path=SEM_PATH,
#     nominal_params={'W1':300,'W2':400,'W3':500,'W4':600,'L1':200,'L2':300},
#     sam_points=[[400,190],[900,190]],   # None -> interactive window
#     sam_checkpoint=SAM_CKPT,
#     design_gds=DESIGN_GDS,              # None -> skip Stage 4
#     out_prefix='results/run1',
# )
# print(result['complete_params'], result['best_hausdorff'])